In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

# 1. Cargar el archivo M17.csv directamente
df = pd.read_csv('M17.csv')

# 2. Separar conjuntos de entrenamiento y prueba según la planilla oficial
train_df = df[df['conjunto'] == 'entrenamiento']
test_df = df[df['conjunto'] == 'prueba']

X_train = train_df[['Distancia del tiro (m)', 'Defensor más cercano (m)', 'Efectividad del tirador (% en triples en la temporada)']]
y_train = train_df['resultado_real']

X_test = test_df[['Distancia del tiro (m)', 'Defensor más cercano (m)', 'Efectividad del tirador (% en triples en la temporada)']]
y_test = test_df['resultado_real']

# 3. Entrenar el modelo (Regresión Logística con castigo asimétrico a Falsos Positivos)
# Penalizamos los errores en la clase 1 (decir SÍ cuando era NO) porque cuestan el partido.
modelo = LogisticRegression(class_weight={0: 1, 1: 6}, random_state=42)
modelo.fit(X_train, y_train)

# 4. Predicción con umbral estricto (0.75) para blindar el sistema ante Falsos Positivos
probabilidades = modelo.predict_proba(X_test)[:, 1]
umbral = 0.75
predicciones = (probabilidades >= umbral).astype(int)

# 5. Mostrar resultados en el conjunto de prueba
resultados = test_df.copy()
resultados['Probabilidad_SI'] = probabilidades
resultados['Prediccion_IA'] = ["SÍ" if p == 1 else "NO" for p in predicciones]
resultados['Resultado_Real'] = ["SÍ" if r == 1 else "NO" for r in y_test.values]

print("--- EVALUACIÓN EN EL CONJUNTO DE PRUEBA (M17.csv) ---")
print(resultados[['caso', 'Distancia del tiro (m)', 'Defensor más cercano (m)', 'Efectividad del tirador (% en triples en la temporada)', 'Probabilidad_SI', 'Prediccion_IA', 'Resultado_Real']].to_string(index=False))

# 6. Mostrar resultados en el conjunto de entrenamiento
probabilidades_train = modelo.predict_proba(X_train)[:, 1]
predicciones_train = (probabilidades_train >= umbral).astype(int)

resultados_train = train_df.copy()
resultados_train['Probabilidad_SI'] = probabilidades_train
resultados_train['Prediccion_IA'] = ["SÍ" if p == 1 else "NO" for p in predicciones_train]
resultados_train['Resultado_Real'] = ["SÍ" if r == 1 else "NO" for r in y_train.values]

print("\n--- EVALUACIÓN EN EL CONJUNTO DE ENTRENAMIENTO (M17.csv) ---")
print(resultados_train[['caso', 'Distancia del tiro (m)', 'Defensor más cercano (m)', 'Efectividad del tirador (% en triples en la temporada)', 'Probabilidad_SI', 'Prediccion_IA', 'Resultado_Real']].to_string(index=False))


--- EVALUACIÓN EN EL CONJUNTO DE PRUEBA (M17.csv) ---
 caso  Distancia del tiro (m)  Defensor más cercano (m)  Efectividad del tirador (% en triples en la temporada)  Probabilidad_SI Prediccion_IA Resultado_Real
   15                    8.87                       2.9                                                      28     9.066672e-07            NO             NO
   16                    8.97                       0.9                                                      35     2.692300e-04            NO             NO
   17                    6.94                       0.6                                                      39     4.058952e-02            NO             NO
   18                    6.76                       1.7                                                      44     9.686542e-01            SÍ             SÍ
   19                    7.19                       2.6                                                      28     1.525769e-06            NO             N

In [4]:
import pandas as pd

# 7. Evaluar nuevos casos inventados
datos_nuevos = {
    'caso': ['A', 'B'],
    'Distancia del tiro (m)': [27, 13],
    'Defensor más cercano (m)': [1, 2],
    'Efectividad del tirador (% en triples en la temporada)': [20, 36.6]
}

nuevos_df = pd.DataFrame(datos_nuevos)

X_nuevos = nuevos_df[['Distancia del tiro (m)', 'Defensor más cercano (m)', 'Efectividad del tirador (% en triples en la temporada)']]

probabilidades_nuevos = modelo.predict_proba(X_nuevos)[:, 1]
predicciones_nuevos = (probabilidades_nuevos >= umbral).astype(int)

resultados_nuevos = nuevos_df.copy()
resultados_nuevos['Probabilidad_SI'] = probabilidades_nuevos
resultados_nuevos['Prediccion_IA'] = ["SÍ" if p == 1 else "NO" for p in predicciones_nuevos]

print("\n--- EVALUACIÓN DE CASOS NUEVOS ---")
print(resultados_nuevos[['caso', 'Distancia del tiro (m)', 'Defensor más cercano (m)', 'Efectividad del tirador (% en triples en la temporada)', 'Probabilidad_SI', 'Prediccion_IA']].to_string(index=False))


--- EVALUACIÓN DE CASOS NUEVOS ---
caso  Distancia del tiro (m)  Defensor más cercano (m)  Efectividad del tirador (% en triples en la temporada)  Probabilidad_SI Prediccion_IA
   A                      27                         1                                                    20.0     3.986846e-15            NO
   B                      13                         2                                                    36.6     6.364985e-04            NO
